In [1]:
import numpy as np
import pandas as pd

# -----------------------------
# HMM Components
# -----------------------------

states = ['h', 'e', 'l', 'o']
observations = ['O1', 'O2', 'O3', 'O4']
obs_seq = ['O1', 'O2', 'O3', 'O4']

state_index = {s: i for i, s in enumerate(states)}
obs_index = {o: i for i, o in enumerate(observations)}

A = np.array([
    [0.7, 0.3, 0.0, 0.0],
    [0.0, 0.2, 0.6, 0.2],
    [0.0, 0.0, 0.3, 0.7],
    [0.0, 0.0, 0.1, 0.9]
])

B = np.array([
    [0.6, 0.2, 0.1, 0.1],
    [0.1, 0.7, 0.1, 0.1],
    [0.1, 0.1, 0.6, 0.2],
    [0.2, 0.1, 0.2, 0.5]
])

pi = np.array([1.0, 0.0, 0.0, 0.0])


In [2]:

# -----------------------------
# Viterbi Algorithm
# -----------------------------

T = len(obs_seq)
N = len(states)

viterbi = np.zeros((N, T))
backpointer = np.zeros((N, T), dtype=int)

# ---- Step 1: Initialization ----
for s in range(N):
    viterbi[s, 0] = pi[s] * B[s, obs_index[obs_seq[0]]]
    backpointer[s, 0] = 0

print("\n=== INITIALIZATION STEP ===")
df_init = pd.DataFrame(viterbi[:, 0], index=states, columns=["t=0"])
print(df_init)

# ---- Step 2: Recursion ----
for t in range(1, T):
    for s in range(N):
        transition_probs = viterbi[:, t-1] * A[:, s]
        best_prev_state = np.argmax(transition_probs)
        viterbi[s, t] = transition_probs[best_prev_state] * B[s, obs_index[obs_seq[t]]]
        backpointer[s, t] = best_prev_state

print("\n=== RECURSION STEP (VITERBI TABLE) ===")
df_viterbi = pd.DataFrame(viterbi, index=states, columns=[f"t={i}" for i in range(T)])
print(df_viterbi)

print("\n=== BACKPOINTER TABLE ===")
df_back = pd.DataFrame(backpointer, index=states, columns=[f"t={i}" for i in range(T)])
print(df_back)

# ---- Step 3: Termination + Backtracking ----
best_last_state = np.argmax(viterbi[:, T-1])
best_path_prob = viterbi[best_last_state, T-1]

best_path = [best_last_state]
for t in range(T-1, 0, -1):
    best_last_state = backpointer[best_last_state, t]
    best_path.insert(0, best_last_state)

decoded_states = [states[i] for i in best_path]



=== INITIALIZATION STEP ===
   t=0
h  0.6
e  0.0
l  0.0
o  0.0

=== RECURSION STEP (VITERBI TABLE) ===
   t=0    t=1      t=2       t=3
h  0.6  0.084  0.00588  0.000412
e  0.0  0.126  0.00252  0.000176
l  0.0  0.000  0.04536  0.002722
o  0.0  0.000  0.00504  0.015876

=== BACKPOINTER TABLE ===
   t=0  t=1  t=2  t=3
h    0    0    0    0
e    0    0    0    0
l    0    0    1    2
o    0    0    1    2


In [3]:

print("\n=== FINAL RESULT ===")
print("Most Likely Phoneme Sequence:", decoded_states)
print("Probability of Best Path:", best_path_prob)



=== FINAL RESULT ===
Most Likely Phoneme Sequence: ['h', 'e', 'l', 'o']
Probability of Best Path: 0.015875999999999998
